In [ ]:
# ============================================
# FINAL: HYSTERESIS (CURRENT vs INTENSITY)
# ============================================

from pypylon import pylon
from DeviceInterface import DeviceInterface
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
import os

# ==============================
# SETTINGS
# ==============================
settle_time = 2.0
SAVE_IMAGES = False   # change to True if needed

folder = "captured_images"
os.makedirs(folder, exist_ok=True)

# ==============================
# INIT CAMERA
# ==============================
camera = pylon.InstantCamera(
    pylon.TlFactory.GetInstance().CreateFirstDevice()
)
camera.Open()

camera.PixelFormat.SetValue("Mono8")

# FIXED EXPOSURE
camera.ExposureAuto.SetValue("Off")
camera.GainAuto.SetValue("Off")

camera.ExposureTime.SetValue(20000.0)   # tune if needed
camera.Gain.SetValue(2.0)

print("Camera initialized with manual exposure")

# ==============================
# INIT MAGNET
# ==============================
Mode = "Constant Current"

DeviceInterface.reset_limit_status()
DeviceInterface.config_serialport("COM3")

# ==============================
# STEP 1: CAPTURE ONCE + SELECT ROI
# ==============================
print("\nCapture initial image for ROI selection")

grab = camera.GrabOne(2000)

if not grab.GrabSucceeded():
    camera.Close()
    raise RuntimeError("Initial capture failed")

img = grab.Array
print("Mean brightness:", np.mean(img))

display = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

roi = cv2.selectROI("Select ROI", display, False, False)
cv2.destroyWindow("Select ROI")

x, y, w, h = roi

if w == 0 or h == 0:
    camera.Close()
    raise RuntimeError("Invalid ROI")

print(f"ROI fixed: x={x}, y={y}, w={w}, h={h}")

# ==============================
# DEFINE HYSTERESIS SWEEP
# ==============================
start = -3.5
stop = 3.5
step = 0.5

forward = np.arange(start, stop + step, step)
reverse = np.arange(stop, start - step, -step)

currents = np.concatenate((forward, reverse))

# ==============================
# DATA STORAGE
# ==============================
I_data = []
Intensity_data = []

# ==============================
# RUN EXPERIMENT
# ==============================
if DeviceInterface.connect():
    DeviceInterface.electromagnet_on()

    try:
        img_count = 0

        for current in currents:

            print(f"\nSetting current: {current:.2f} A")
            DeviceInterface.set_field(Mode, current)

            # wait for stabilization
            time.sleep(settle_time)

            # capture image
            grab = camera.GrabOne(2000)

            if not grab.GrabSucceeded():
                print("Capture failed")
                continue

            img = grab.Array

            # compute intensity from FIXED ROI
            roi_img = img[y:y+h, x:x+w]
            avg_intensity = np.mean(roi_img)

            print(f"→ Avg Intensity: {avg_intensity:.2f}")

            # store data
            I_data.append(current)
            Intensity_data.append(avg_intensity)

            # optional save
            if SAVE_IMAGES:
                filename = f"{folder}/img_{img_count:03d}_{current:.2f}A.png"
                cv2.imwrite(filename, img)
                img_count += 1

            # show ROI overlay (optional visual)
            display = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            cv2.rectangle(display, (x, y), (x+w, y+h), (0,255,0), 2)

            cv2.putText(display,
                        f"{avg_intensity:.1f}",
                        (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0,255,0),
                        2)

            cv2.imshow("Measurement", display)
            cv2.waitKey(200)

    finally:
        DeviceInterface.electromagnet_off()
        DeviceInterface.clear_comports()
        camera.Close()
        cv2.destroyAllWindows()

        print("\nExperiment complete")

# ==============================
# FINAL HYSTERESIS PLOT
# ==============================
mid = len(I_data) // 2

I_forward = I_data[:mid]
I_reverse = I_data[mid:]

Int_forward = Intensity_data[:mid]
Int_reverse = Intensity_data[mid:]

plt.figure(figsize=(7,5))

plt.plot(I_forward, Int_forward, 'bo-', label="Forward Sweep")
plt.plot(I_reverse, Int_reverse, 'ro-', label="Reverse Sweep")

plt.xlabel("Current (A)")
plt.ylabel("Avg Pixel Intensity")
plt.title("Hysteresis: Current vs Intensity")
plt.legend()
plt.grid(True)

plt.show()